<a href="https://colab.research.google.com/github/NadiiaStuzhuk/DI-Bootcamp-Stage1/blob/main/%22Exercises_XP_Day3_Student_NS_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "TODO: replace with a short sentence you want to tokenize"
print(sample_sentence)


In [2]:
# encoding = tokenizer(
#     sample_sentence,
#     add_special_tokens=True,
#     padding="max_length",
#     truncation=True,
#     max_length=24,  # TODO: adjust if your sentence needs more room
#     return_attention_mask=True,
#     return_tensors="pt"
# )

# input_ids = encoding["input_ids"][0].tolist()
# tokens = tokenizer.convert_ids_to_tokens(input_ids)
# print("index | token        | id")
# print("-------------------------")
# for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
#     print(f"{idx:>5} | {token:<12} | {token_id:>5}")

# print("\nAttention mask:", encoding["attention_mask"][0].tolist())
# special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
# print("Special tokens (index, token):", special_positions)





# 1. Install the transformers and torch libraries (uncomment if running in Colab)
# !pip install -q transformers torch

import torch
from transformers import AutoTokenizer

# 2. Load the BERT tokenizer (bert-base-uncased)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 3. Choose a sample sentence
sample_sentence = "BERT transforms natural language processing into an exciting journey!"

# 4. Tokenize the sentence and view how BERT breaks it down (WordPiece subwords)
raw_tokens = tokenizer.tokenize(sample_sentence)
print("Raw Tokens:", raw_tokens)

# 5. Prepare the sentence with special tokens, padding, and truncation for model input
encoded_input = tokenizer(
    sample_sentence,
    add_special_tokens=True,  # Adds [CLS] at start and [SEP] at end
    max_length=16,            # Fixed sequence length
    padding="max_length",     # Pads shorter sequences with [PAD] (ID: 0)
    truncation=True,          # Truncates sequences longer than max_length
    return_tensors="pt"       # Returns PyTorch tensors
)

# 6. Review the token IDs, attention mask, and reconstructed tokens
input_ids = encoded_input["input_ids"][0]
attention_mask = encoded_input["attention_mask"][0]
tokens_with_special = tokenizer.convert_ids_to_tokens(input_ids)

print("\n--- Tokenization Breakdown ---")
print(f"{'Index':<6} | {'Token':<14} | {'Token ID':<10} | {'Attention Mask'}")
print("-" * 50)
for idx, (tok, tid, mask) in enumerate(zip(tokens_with_special, input_ids.tolist(), attention_mask.tolist())):
    print(f"{idx:<6} | {tok:<14} | {tid:<10} | {mask}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Raw Tokens: ['bert', 'transforms', 'natural', 'language', 'processing', 'into', 'an', 'exciting', 'journey', '!']

--- Tokenization Breakdown ---
Index  | Token          | Token ID   | Attention Mask
--------------------------------------------------
0      | [CLS]          | 101        | 1
1      | bert           | 14324      | 1
2      | transforms     | 21743      | 1
3      | natural        | 3019       | 1
4      | language       | 2653       | 1
5      | processing     | 6364       | 1
6      | into           | 2046       | 1
7      | an             | 2019       | 1
8      | exciting       | 10990      | 1
9      | journey        | 4990       | 1
10     | !              | 999        | 1
11     | [SEP]          | 102        | 1
12     | [PAD]          | 0          | 0
13     | [PAD]          | 0          | 0
14     | [PAD]          | 0          | 0
15     | [PAD]          | 0          | 0


### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.


Key Role of Special Tokens:

[CLS] (ID 101): Placed at the very beginning of every input sequence. Its final hidden state acts as the aggregate sequence representation for classification tasks.

[SEP] (ID 102): Marks the end of a single sentence or separates two distinct sentences in pair-level tasks.

[PAD] (ID 0): Fills remaining positions up to max_length so batches form uniform rectangular tensors. The attention_mask sets 0 for [PAD] tokens so self-attention ignores them.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [4]:
# from transformers import pipeline

# sentiment_pipeline = pipeline(
#     task="sentiment-analysis",
#     model="distilbert-base-uncased-finetuned-sst-2-english"
# )

# sentence = "TODO: add a sentence whose sentiment you want to test"
# prediction = sentiment_pipeline(sentence)
# prediction


from transformers import pipeline

# 1 & 2. Create a sentiment analysis pipeline using the pre-trained DistilBERT SST-2 model
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# 3. Provide sample sentences
sample_sentences = [
    "The new data pipeline reduced our processing time by 70%—it works brilliantly!",
    "The application kept crashing during the presentation and lost all my saved configurations."
]

# 4 & 5. Predict sentiment and review labels and confidence scores
results = sentiment_pipeline(sample_sentences)

print("--- Sentiment Pipeline Results ---")
for text, res in zip(sample_sentences, results):
    print(f"Text      : {text}")
    print(f"Prediction: {res['label']} (Confidence: {res['score']:.4f})\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- Sentiment Pipeline Results ---
Text      : The new data pipeline reduced our processing time by 70%—it works brilliantly!
Prediction: POSITIVE (Confidence: 0.9998)

Text      : The application kept crashing during the presentation and lost all my saved configurations.
Prediction: NEGATIVE (Confidence: 0.9998)



### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [5]:
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# from typing import Dict

# class BERTSentimentAnalyzer:
#     def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
#         '''TODO: load the tokenizer/model and move the model to the proper device.'''
#         raise NotImplementedError("Initialize tokenizer, model, and device here.")

#     def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
#         '''TODO: clean the text, tokenize, and return tensors ready for inference.'''
#         raise NotImplementedError("Return a dict of tensors produced by the tokenizer.")

#     def predict(self, text: str) -> Dict[str, float]:
#         '''TODO: run a forward pass, apply softmax, and return a label plus probability.'''
#         raise NotImplementedError("Add inference and post-processing logic.")



import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class BERTSentimentAnalyzer:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        """1. Initialize tokenizer, model, and computation device."""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set model to evaluation mode

    def preprocess(self, text, max_length=128):
        """2. Clean text, tokenize, and prepare PyTorch tensors."""
        # Basic text normalization: strip extra whitespace
        cleaned_text = re.sub(r"\s+", " ", text).strip()

        inputs = self.tokenizer(
            cleaned_text,
            add_special_tokens=True,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {k: v.to(self.device) for k, v in inputs.items()}

    def predict(self, text):
        """3. Run forward pass, compute softmax probabilities, and return structured output."""
        inputs = self.preprocess(text)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1).squeeze()

        pred_idx = torch.argmax(probs).item()
        label = self.model.config.id2label[pred_idx]
        confidence = probs[pred_idx].item()

        return {
            "text": text,
            "label": label,
            "confidence": round(confidence, 4),
            "probabilities": {
                self.model.config.id2label[i]: round(probs[i].item(), 4)
                for i in range(len(probs))
            }
        }

# Test the custom analyzer with various sample texts
analyzer = BERTSentimentAnalyzer()
test_texts = [
    "This tutorial explains transformer architectures with crystal clarity!",
    "The customer support was unhelpful and the documentation is completely outdated.",
    "The package arrived on Tuesday as scheduled."
]

print("--- Custom BERTSentimentAnalyzer Output ---")
for sample in test_texts:
    output = analyzer.predict(sample)
    print(f"Input : {output['text']}")
    print(f"Result: {output['label']} (Confidence: {output['confidence']}) | Full Probs: {output['probabilities']}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- Custom BERTSentimentAnalyzer Output ---
Input : This tutorial explains transformer architectures with crystal clarity!
Result: POSITIVE (Confidence: 0.9999) | Full Probs: {'NEGATIVE': 0.0001, 'POSITIVE': 0.9999}

Input : The customer support was unhelpful and the documentation is completely outdated.
Result: NEGATIVE (Confidence: 0.9998) | Full Probs: {'NEGATIVE': 0.9998, 'POSITIVE': 0.0002}

Input : The package arrived on Tuesday as scheduled.
Result: POSITIVE (Confidence: 0.8099) | Full Probs: {'NEGATIVE': 0.1901, 'POSITIVE': 0.8099}



In [ ]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
# analyzer = BERTSentimentAnalyzer()
# samples = [
#     "TODO: add a clearly positive statement",
#     "TODO: add a clearly negative statement"
# ]
# for text in samples:
#     print(text)
#     print(analyzer.predict(text))


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [6]:
# from transformers import AutoTokenizer, AutoModelForTokenClassification
# import torch

# class BERTNamedEntityRecognizer:
#     def __init__(self, model_name: str = "dslim/bert-base-NER"):
#         '''TODO: load the tokenizer and model, and detect the available device.'''
#         raise NotImplementedError("Initialize tokenizer, model, and device.")

#     def recognize(self, text: str):
#         '''TODO: tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
#         raise NotImplementedError("Return structured entities.")


import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

class BERTNamedEntityRecognizer:
    def __init__(self, model_name="dslim/bert-base-NER"):
        """1. Initialize the tokenizer and token classification model."""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def recognize_entities(self, text):
        """2. Identify entities, map token IDs to B-I-O labels, and merge subwords."""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            return_offsets_mapping=True
        )
        offset_mapping = inputs.pop("offset_mapping")[0].tolist()
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()

        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        id2label = self.model.config.id2label

        token_results = []
        entities = []
        current_entity = None

        for token, pred_id, (start, end) in zip(tokens, predictions, offset_mapping):
            # Skip special tokens ([CLS], [SEP], [PAD])
            if start == end:
                continue

            label = id2label[pred_id]
            token_results.append((token, label))

            # Group B-I-O tags and merge WordPiece subwords (##)
            if label.startswith("B-"):
                if current_entity:
                    entities.append(current_entity)
                current_entity = {
                    "entity_type": label[2:],
                    "word": text[start:end],
                    "start": start,
                    "end": end
                }
            elif label.startswith("I-") and current_entity and current_entity["entity_type"] == label[2:]:
                current_entity["end"] = end
                current_entity["word"] = text[current_entity["start"]:end]
            else:
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        if current_entity:
            entities.append(current_entity)

        return token_results, entities

# 3. Test recognizer with sample text containing entities
ner_recognizer = BERTNamedEntityRecognizer()
sample_bio_text = "Sundar Pichai announced that Google is expanding its AI research center in Tel Aviv and Paris."

token_tags, extracted_entities = ner_recognizer.recognize_entities(sample_bio_text)

print("--- Token-Level B-I-O Tags ---")
print([f"{tok}: {tag}" for tok, tag in token_tags if tag != "O"])

print("\n--- Merged Named Entities ---")
for ent in extracted_entities:
    print(f"Entity: {ent['word']:<15} | Type: {ent['entity_type']:<5} | Span: ({ent['start']}, {ent['end']})")


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Token-Level B-I-O Tags ---
['Sun: B-PER', '##dar: I-PER', 'Pi: I-PER', '##cha: I-PER', '##i: I-PER', 'Google: B-ORG', 'AI: B-MISC', 'Tel: B-LOC', 'Aviv: I-LOC', 'Paris: B-LOC']

--- Merged Named Entities ---
Entity: Sundar Pichai   | Type: PER   | Span: (0, 13)
Entity: Google          | Type: ORG   | Span: (29, 35)
Entity: AI              | Type: MISC  | Span: (53, 55)
Entity: Tel Aviv        | Type: LOC   | Span: (75, 83)
Entity: Paris           | Type: LOC   | Span: (88, 93)


In [ ]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
# ner = BERTNamedEntityRecognizer()
# sample_text = "TODO: add a short paragraph with at least two entities."
# ner.recognize(sample_text)


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | TODO | TODO |
| Primary purpose | TODO | TODO |
| Typical use cases | TODO | TODO |
| Strengths | TODO | TODO |
| Weaknesses | TODO | TODO |

Feature / Dimension,BERT (Bidirectional Encoder Representations from Transformers),GPT (Generative Pre-trained Transformer)
Architecture,Transformer Encoder-only stack.,Transformer Decoder-only stack.
Attention Mechanism,Bidirectional self-attention: each token attends to both left and right context simultaneously.,Causal (masked) self-attention: each token only attends to preceding tokens to its left.
Pre-training Objective,Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).,Autoregressive Next-Token Prediction (Causal Language Modeling).
Primary Purpose,Natural Language Understanding (NLU) and deep contextual representation.,Natural Language Generation (NLG) and zero/few-shot task completion.
Common Use Cases,"Sentiment analysis, Named Entity Recognition (NER), semantic search embeddings, extractive question answering, text classification.","Conversational agents, code generation, summarization, creative writing, translation, open-ended reasoning."
Strengths,Captures rich bidirectional context; compact and fast to fine-tune for discriminative classification or retrieval tasks.,"Generates fluent, coherent long-form text; versatile across diverse tasks via prompting without task-specific architecture changes."
Weaknesses,Not designed for free-form text generation.,Unidirectional context makes pure token embeddings less optimal for compact bi-directional classification at equivalent small parameter scales.


Reflection on Differences and Similarities:

Similarities: Both models are built on the Transformer self-attention mechanism, rely on subword tokenization, and use self-supervised pre-training on massive text corpora followed by task adaptation.

Core Difference: BERT acts as an analytical reader that inspects an entire sentence at once to extract meaning or classify tokens, whereas GPT acts as an autoregressive writer that predicts one token after another to generate responses.



## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.


Exercise 6: Exploring BERT Applications in Retrieval-Augmented Generation (RAG)
1. What is Retrieval-Augmented Generation (RAG)?
Retrieval-Augmented Generation (RAG) is a hybrid architecture that connects a retriever (which searches an external knowledge base) with a generator (an autoregressive LLM such as GPT). Instead of relying solely on static pre-trained weights, RAG fetches relevant, up-to-date facts from verified documents and injects them into the generator's prompt to ground the final response and reduce hallucinations.

2. BERT's Role in the Retrieval Component
In a RAG pipeline, BERT (or a fine-tuned bi-encoder variant such as Sentence-BERT / DPR / BGE) serves as the dense embedding model and/or cross-encoder reranker:

Dense Retriever (Bi-Encoder): Converts both user queries and external knowledge chunks into fixed-size semantic vectors in a shared vector space.

Reranker (Cross-Encoder): Takes the top-
𝑘
k candidate passages alongside the query ([CLS] Query [SEP] Passage [SEP]) and computes a fine-grained relevance score to order the best matches.

3. How BERT Generates Embeddings for Documents and Queries
Document Ingestion: External documents are split into manageable chunks (e.g., 256–512 tokens) and passed through BERT.

Pooling: The contextualized token representations from BERT's final layer are aggregated—most commonly via mean pooling across all non-padded tokens or by extracting the [CLS] token vector—and normalized to produce a single dense vector (e.g.,
768
768 dimensions) per chunk.

Query Encoding: When a user submits a question, the exact same BERT encoder transforms the query into a
768
768-dimensional vector in the same semantic space.

4. Matching Queries with Documents in a Vector Database
All pre-computed document embeddings are indexed in a Vector Database (such as FAISS, Pinecone, Qdrant, or Chroma) using Approximate Nearest Neighbor (ANN) structures like HNSW (Hierarchical Navigable Small World). When the query vector arrives, the database computes similarity scores—such as Cosine Similarity (
cos
⁡
(
𝜃
)
=
𝑞
⋅
𝑑
∥
𝑞
∥
∥
𝑑
∥
cos(θ)=
∥q∥∥d∥
q⋅d
​
 ) or Inner Product—and returns the top-
𝑘
k most semantically relevant passages in milliseconds, even when the query and document use different synonyms.

5. Concrete Example: BERT + GPT Working Together in a RAG System
User Query: "What is our company's refund policy for annual enterprise subscriptions?"

BERT Encoding: A BERT-based encoder converts the question into a query vector
𝑣
𝑞
v
q
​
 .

Vector Search: The vector database compares
𝑣
𝑞
v
q
​
  against millions of company policy chunks and retrieves the top passage: "Policy 4.2: Annual enterprise subscriptions are eligible for a prorated refund within 30 days of renewal."

Prompt Augmentation: The system constructs a grounded prompt:

Plaintext
Context: Policy 4.2: Annual enterprise subscriptions are eligible for a prorated refund within 30 days of renewal.
Question: What is our company's refund policy for annual enterprise subscriptions?
Answer based only on the context:
GPT Generation: GPT reads the retrieved context and generates a fluent, accurate answer: "Annual enterprise subscriptions qualify for a prorated refund if requested within 30 days of the renewal date."